<img src="../assets/agent_patterns_03_overview.png" alt="A2A Handoff 흐름" style="display:block;width:100%;max-width:1200px;margin:0 auto 24px;border-radius:12px;">


![LLM-Orchestrated A2A Workflow](../assets/workflow_diagrams/agent_to_agent.svg)


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #2e7d32; letter-spacing: .02em;">🖥️ A2A Chat UI</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">먼저 별도 터미널에서 <code>python 4_agent_patterns/main.py fnb-a2a-local</code>을 실행하세요. 이 노트북은 기본 A2A 협업 Chat UI를 실행합니다.</div>
</div>


In [ ]:
from pathlib import Path
import importlib
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '4_agent_patterns' / 'src').is_dir())
if str(project_root / '4_agent_patterns') not in sys.path:
    sys.path.insert(0, str(project_root / '4_agent_patterns'))
import src.agent_to_agent.catalog as a2a_catalog_module
import src.agent_to_agent.orchestration as a2a_module
import src.agent_to_agent.ui as a2a_ui_module
a2a_catalog_module = importlib.reload(a2a_catalog_module)
a2a_module = importlib.reload(a2a_module)
a2a_ui_module = importlib.reload(a2a_ui_module)
DEFAULT_REQUEST = a2a_catalog_module.DEFAULT_REQUEST
SCENARIO_REQUESTS = a2a_catalog_module.SCENARIO_REQUESTS
list_agents = a2a_module.list_agents
run_handoff = a2a_module.run_handoff
if 'a2a_ui' in globals():
    a2a_ui.close()
a2a_ui = a2a_ui_module.launch_a2a_ui()

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅ 체크포인트</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">A2A Team Card, Task ID, TextPart·JSON DataPart, 사람 승인 및 외부 시스템 쓰기 차단 경계를 Chat UI에서 확인합니다.</div></div>


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
    <h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">언제 A2A를 사용할까?</h2>
    <p style="margin: 7px 0;">고정된 내부 Agent를 한 팀이 관리한다면 일반 HTTPS delegation만으로 충분합니다. URL, 요청·응답 JSON, 인증, 재시도, 작업 상태를 팀의 API 계약으로 일관되게 관리하면 A2A는 필수가 아닙니다.</p>
    <table style="width:100%; margin:10px 0 4px; border-collapse:collapse; font-size:.92em;"><thead><tr><th style="text-align:left; padding:7px; border-bottom:1px solid #c7d2e1;">필요</th><th style="text-align:left; padding:7px; border-bottom:1px solid #c7d2e1;">A2A 기능</th></tr></thead><tbody><tr><td style="padding:7px; border-bottom:1px solid #d9e2f3;">무슨 Agent인지·무엇을 하는지 발견</td><td style="padding:7px; border-bottom:1px solid #d9e2f3;"><b>Agent Card</b>의 Skill, 입출력 모드, 접속 URL</td></tr><tr><td style="padding:7px; border-bottom:1px solid #d9e2f3;">장기 작업의 상태·실패·취소 관리</td><td style="padding:7px; border-bottom:1px solid #d9e2f3;"><b>Task</b> ID와 working / completed / failed 등 표준 lifecycle</td></tr><tr><td style="padding:7px; border-bottom:1px solid #d9e2f3;">다음 Agent에 구조화 결과 전달</td><td style="padding:7px; border-bottom:1px solid #d9e2f3;"><b>Message·Artifact·DataPart</b>의 JSON·파일 전달 계약</td></tr><tr><td style="padding:7px;">서로 다른 팀·프레임워크 Agent 연결</td><td style="padding:7px;"><b>JSON-RPC/HTTP</b> 공통 계약으로 내부 구현을 숨긴 상호운용</td></tr></tbody></table>
    <p style="margin: 9px 0 0; color:#4b5f78;"><b>판단:</b> 내부의 고정된 3개 Agent라면 HTTPS handoff가 더 단순할 수 있습니다. 외부·타팀 Agent, 교체 가능한 Agent, 여러 프레임워크 연결이 필요할 때 A2A의 표준화 비용 절감 효과가 커집니다.</p>
  </div>
</div>


In [16]:
from src.agent_to_agent.catalog import DEFAULT_REQUEST
from src.agent_to_agent.orchestration import list_agents, run_handoff_steps


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. A2A Team Card와 내부 sub-agent 보기</h2><p style="margin: 7px 0;">A2A는 조직 경계만 공개합니다. 설비팀 안에는 현장·엔지니어·관리자 Agent, 구매팀 안에는 구매·원가·재고·MDM Agent가 있습니다. 제조기획팀 Assistant는 생산기획·OPEX 예산·Cost Center 관리·예산 이관 신청 Agent를, IT팀 Assistant는 ERP 데이터·IT 변경관리 Agent를 조율합니다.</p></div></div>


In [17]:
for agent in list_agents():
    print(f"## {agent['agent_label']} ({agent['agent_id']})")
    print(f"설명: {agent['description']}")
    print(f"Skill: {agent['skill_id']} | URL: {agent['url']}")
    print(f"내부 sub-agent: {' → '.join(agent['sub_agents'])}")
    print(f"실행 경계: {agent['execution_boundary']}\n")


## F&B 설비팀 Assistant (fb-facilities-team)
설명: 설비 식별자·현장 관찰·기술 제약·운영 영향을 구조화합니다. 조달·재무·데이터 변경의 판단은 해당 책임 Team의 검토 범위입니다.
Skill: facility_incident_and_recovery | URL: http://127.0.0.1:9111
내부 sub-agent: 현장 운영 Agent → 설비기술 Agent
실행 경계: 모든 업무 판단·delegation·handoff와 필요 시 SAP BTP Grounding Tool 호출은 현장 요청과 누적 artifact를 바탕으로 AI Core LLM이 생성합니다. 코드는 Tool 실행, A2A 전송, 등록 Agent 검증, 반복 한도와 외부 쓰기 차단만 담당합니다.

## F&B 구매팀 Assistant (fb-procurement-team)
설명: 조달·재고·수급 관점의 선택지를 검토합니다. 역할별 읽기 전용 Tool 결과가 있으면 그 범위 안에서만 근거로 사용합니다.
Skill: procurement_and_supply_review | URL: http://127.0.0.1:9112
내부 sub-agent: 구매조달 Agent → 재고·수급 Agent
실행 경계: 모든 업무 판단·delegation·handoff와 필요 시 SAP BTP Grounding Tool 호출은 현장 요청과 누적 artifact를 바탕으로 AI Core LLM이 생성합니다. 코드는 Tool 실행, A2A 전송, 등록 Agent 검증, 반복 한도와 외부 쓰기 차단만 담당합니다.

## 제조기획팀 Assistant (manufacturing-planning-team)
설명: 생산계획, 자재소요, 신규 원부자재와 생산 투입 준비도를 검토합니다. 역할별 읽기 전용 CSV 스냅샷 결과가 있으면 그 범위 안에서만 근거로 사용합니다.
Skill: production_and_material_readiness | URL: http://127.0.0.1:9113


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. Cell 단위 A2A Handoff</h2><p style="margin: 7px 0;">현장 handoff, Orchestrator 판단, Team Agent의 A2A Task·Artifact, 요청자 업데이트를 각각 별도 셀에서 확인합니다. 로컬 A2A 서버는 백그라운드에서 동작하지만, 노트북은 증거를 Team 단위로 한 번에 하나씩 소비합니다.</p></div></div>


In [18]:
request = DEFAULT_REQUEST  # 필요하면 여기에서 실제 현장 장애 설명으로 바꾸세요.
a2a_steps = run_handoff_steps(request)
result = None
print('A2A handoff session을 준비했습니다. 다음 셀부터 한 단위씩 진행하세요.')


A2A handoff session을 준비했습니다. 다음 셀부터 한 단위씩 진행하세요.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. 현장 handoff와 Orchestrator 관찰</h2><p style="margin: 7px 0;">현장 Agent의 A2A handoff envelope와 Orchestrator 관찰을 각각 한 셀에서 봅니다. Grounding·Decision·Task·Artifact는 다음 이벤트 셀에서 하나씩 확인합니다.</p></div></div>


In [19]:
field_step = next(a2a_steps)
assert field_step['step'] == 'field_handoff', field_step['step']
print(field_step['handoff'])


{'source_agent_id': 'field-agent', 'target_agent_id': 'orchestration-agent', 'purpose': 'field_incident_handoff', 'request': 'CIP 펌프 메커니컬 실 키트를 구매해 주세요.'}


In [20]:
observation_step = next(a2a_steps)
assert observation_step['step'] == 'observation', observation_step['step']
print('[Observation]', observation_step['observation'])


[Observation] {'completed_work': ['구매 요청이 접수되었습니다.'], 'new_evidence': ['현장 요청에 CIP 펌프 메커니컬 실 키트 구매가 필요하다는 내용이 있습니다.'], 'unresolved_decisions': ['구매 검토를 위해 펌프 식별정보, 키트 사양, 수량과 희망 납기가 확인되어야 합니다.'], 'available_capabilities': ['설비팀은 설비 식별자와 현장 기술 제약을 검토할 수 있습니다.', '구매팀은 확인된 사양을 바탕으로 조달·재고·수급 선택지를 검토할 수 있습니다.'], 'decision_focus': ['대상 펌프와 교체 부품 사양을 확정한 뒤 구매 가능 여부와 납기를 판단해야 합니다.'], 'chat_summary': 'CIP 펌프 메커니컬 실 키트 구매 요청을 접수했으며, 대상 펌프와 사양·수량·희망 납기 확인이 필요합니다.'}


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">4. 다음 A2A 이벤트 1회 진행</h2><p style="margin: 7px 0;">이 셀을 실행할 때마다 Grounding audit, Decision, Team 시작, Task 상태, Artifact, 요청자 업데이트 또는 완료 중 하나를 확인합니다. <code>complete</code>가 나올 때까지 필요할 때만 다시 실행하세요.</p></div></div>


In [21]:
import json

if result is None:
    next_step = next(a2a_steps)
    if next_step['step'] == 'grounding':
        print('[Grounding audit]', next_step['audit'])
    elif next_step['step'] == 'decision':
        print('[Orchestrator Decision]', next_step['decision'])
    elif next_step['step'] == 'human_approval':
        print('[Human approval draft]', next_step['draft'])
    elif next_step['step'] == 'team_handoff_started':
        print(f"[Team handoff 시작] {next_step['team_id']}")
    elif next_step['step'] == 'task_update':
        task = next_step['task']
        status = task.get('status', {})
        parts = status.get('message', {}).get('parts', [{}])
        raw_status = parts[0].get('text', '') if parts else ''
        try:
            task_event = json.loads(raw_status)
        except (TypeError, json.JSONDecodeError):
            task_event = {}
        event_name = task_event.get('event', 'task_status')
        actor = task_event.get('agent') or task_event.get('team') or next_step['team_id']
        print(f"[Task] {next_step['team_id']} · {actor}: {event_name} · {status.get('state', 'unknown')}")
    elif next_step['step'] == 'artifact':
        record = next_step['record']
        payload = record.get('artifact', {}).get('payload', {})
        synthesis = payload.get('team_synthesis', {})
        print(f"[Artifact] {record['producer']}: {synthesis.get('summary', 'Artifact 수신 완료')}")
    elif next_step['step'] == 'requester_update':
        print('[요청자 업데이트]', next_step['message'])
    elif next_step['step'] == 'complete':
        result = next_step['result']
        print('[complete]', result['termination'])
        print('context_id:', result['context_id'])
    else:
        print(next_step)

[Grounding audit] {'state': 'grounded', 'repository_id': 'f99a04f7-8f58-43cf-8d1a-398dfcb00b70', 'document_count': 23, 'document_ids': ['work_catalog.md', 'roles.md', '01_field_operations_check.md', '02_preventive_maintenance_review.md', '03_inventory_supply_review.md', '11_equipment_incident_review.md', '12_maintenance_purchase_request.md', '18_process_change_yield_review.md', 'collaboration_sop.md']}


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #7b1fa2; background: #f5eef8; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #7b1fa2; letter-spacing: .02em;">📝 Grounding SOP 변경 실습</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">5. SOP를 교체하고 새 요청으로 근거 변화를 확인하기</h2><p style="margin: 7px 0;">아래 셀은 로컬 원본을 바꾸지 않고 메모리에서 SOP를 수정합니다. 이어서 교육용 Vector Grounding 저장소의 기존 SOP를 수정본으로 교체합니다. 인덱싱 반영 후 위 A2A Chat UI에서 같은 설비 구매 요청을 새로 보내, 구매팀 → 설비팀 확인 뒤 Human-in-the-loop 확인이 나오는지 확인하세요. Team 선택과 순서는 코드가 아닌 LLM의 문서 근거 판단입니다.</p><p style="margin: 7px 0 0; color: #8a1c1c;"><b>주의:</b> 교체·복구 셀은 실제 Grounding 저장소 상태를 변경합니다. 교육용 저장소에서만 실행하고, 실습 후 반드시 원본 복구 셀을 실행하세요.</p></div></div>


In [22]:
from src.agent_to_agent import sop_grounding

# 이 텍스트만 바꾸면 다음 실행에서 Grounding SOP 변경안으로 업로드됩니다.
REVISED_SOP_PROCESS = '''## 프로세스

1. 요청 접수
   - 요청 부서, 업무 대상, 요청 목적과 현재 관찰 사실을 구분해 기록한다.
   - 제공되지 않은 사양, 재고, 단가, 예산, 승인 상태는 추정하지 않고 추가 확인 항목으로 남긴다.

2. 현장 작업자 구매 요청의 구매·설비 확인 선행
   - 현장 작업자가 설비·설비 부품·유지보수와 관련된 구매를 요청한 경우, 구매팀 → 설비팀 순서로 확인한다.
   - 먼저 구매팀이 기준정보, 재고·수급, 공급 조건과 구매 요청 상태를 확인한다.
   - 다음으로 설비팀이 구매팀 확인 내용을 바탕으로 대상 식별, 운전 영향, 안전·기술상 제약과 필요한 기술 확인 사항을 검토한다.
   - 사양 등 기술 입력이 아직 없으면 각 Team의 미확정 확인 항목으로 남기며, 이 단계에서 사람의 승인·결정을 요청하지 않는다.
   - 두 Team의 확인 근거가 누적된 후에만 결과를 정리하고 Human-in-the-loop 확인 단계로 넘긴다. 이 순서는 확인과 근거 축적의 기준일 뿐 승인 또는 구매 실행 자체를 뜻하지 않는다.

3. 관련 부서 협업
   - 위 구매 요청 유형에서 구매팀과 설비팀의 확인은 2항의 순서와 범위에 따라 수행한다.
   - 재무·사업기획팀은 예산, 원가, 수익성 또는 이관 검토가 필요한 경우 근거와 한계를 구분한다.
   - IT팀은 승인된 변경이 ERP·MDM 데이터나 변경 통제에 미치는 영향을 검토한다.
   - Orchestration Agent는 누적 근거와 미해결 판단을 관찰해 다음 검토를 요청한다. 이 SOP가 현재 요청에 적용되면 문서에 명시된 순서를 따르며, 해당 순서를 코드나 일반 키워드로 고정하지 않는다.

4. 결과 정리 및 Human-in-the-loop 확인
   - 구매팀·설비팀의 확인 근거와 미확정 항목을 모두 누적한 뒤, 다음 조치를 요청 부서에 전달한다.
   - 그 후 공식 제출, 예산 배정, 승인, 외부 발송 또는 시스템 쓰기가 필요하면 권한 있는 사람이 최종 검토·결정한다.
   - 이 Human-in-the-loop 확인은 구매팀·설비팀 확인이 끝난 뒤의 최종 단계이며, 중간 확인의 미확정 입력만으로 앞당기지 않는다.
'''

# 원복은 노트북 텍스트가 아닌 저장소의 기존 SOP 파일을 다시 업로드합니다.
sop_change = sop_grounding.prepare_default_sop_replacement(REVISED_SOP_PROCESS)
print(f"원복 원본 파일: {sop_change.source_path}")
print(f"교체 대상 SOP: {sop_change.filename} (Grounding resource ID: {sop_change.original_resource_id})")
print(sop_grounding.preview_sop_replacement(sop_change))


원복 원본 파일: /Users/I781176/Desktop/devday2026/data/context/collaboration_sop.md
교체 대상 SOP: collaboration_sop.md (Grounding resource ID: 9a04855d-c93e-43a1-a0ec-48565bcc3f24)
--- 복구할 SOP
+++ 업로드할 수정 SOP
@@ -10,24 +10,23 @@
    - 요청 부서, 업무 대상, 요청 목적과 현재 관찰 사실을 구분해 기록한다.
    - 제공되지 않은 사양, 재고, 단가, 예산, 승인 상태는 추정하지 않고 추가 확인 항목으로 남긴다.
 
-2. 현장 작업자 구매 요청의 설비·구매·제조기획 검토 선행
-   - 현장 작업자가 설비·설비 부품·유지보수와 관련된 구매를 요청한 경우, 설비팀 → 구매팀 → 제조기획팀 순서로 검토한다.
-   - 먼저 설비팀이 현장 관찰 내용, 대상 식별, 운전 영향, 안전·기술상 제약과 필요한 기술 확인 사항을 검토한다.
-   - 다음으로 구매팀이 설비팀 검토 근거를 바탕으로 기준정보, 재고·수급, 공급 조건과 구매 요청 상태를 검토한다.
-   - 이어서 제조기획팀이 구매팀 검토 근거를 바탕으로 생산계획, 자재·투입 준비도, 라인 또는 운영 영향과 필요한 확인 사항을 검토한다.
+2. 현장 작업자 구매 요청의 구매·설비 확인 선행
+   - 현장 작업자가 설비·설비 부품·유지보수와 관련된 구매를 요청한 경우, 구매팀 → 설비팀 순서로 확인한다.
+   - 먼저 구매팀이 기준정보, 재고·수급, 공급 조건과 구매 요청 상태를 확인한다.
+   - 다음으로 설비팀이 구매팀 확인 내용을 바탕으로 대상 식별, 운전 영향, 안전·기술상 제약과 필요한 기술 확인 사항을 검토한다.
    - 사양 등 기술 입력이 아직 없으면 각 Team의 미확정 확인 항목으로 남기며, 이 단계에서 사람의 승인·결정을 요청하지 않는다.
-   - 세 Team의 검토 근거가 모두 누적된 후에만 결과를 정리하고 Huma

In [23]:
print('다음 셀은 위 diff의 수정 SOP를 교육용 Grounding 저장소에 교체 업로드합니다.')
print('실습이 끝나면 마지막 복구 셀을 실행하세요.')


다음 셀은 위 diff의 수정 SOP를 교육용 Grounding 저장소에 교체 업로드합니다.
실습이 끝나면 마지막 복구 셀을 실행하세요.


In [24]:
sop_upload_result = sop_grounding.replace_sop(sop_change)
print('수정 SOP의 Grounding 인덱싱을 요청했습니다:', sop_upload_result)
print('인덱싱 반영에는 잠시 시간이 걸릴 수 있습니다. 반영 후 위 A2A Chat UI에서 같은 설비 구매 요청을 새로 보내세요.')
print('서버 재시작은 필요하지 않습니다. 실습이 끝나면 마지막 복구 셀을 실행하세요.')


수정 SOP의 Grounding 인덱싱을 요청했습니다: {'documents': [{'metadata': [{'key': 'id', 'value': ['collaboration_sop.md'], 'matchMode': 'ANY'}], 'id': '640b81ba-d6cd-4e04-a1b3-02693218a91a'}]}
인덱싱 반영에는 잠시 시간이 걸릴 수 있습니다. 반영 후 위 A2A Chat UI에서 같은 설비 구매 요청을 새로 보내세요.
서버 재시작은 필요하지 않습니다. 실습이 끝나면 마지막 복구 셀을 실행하세요.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #7b1fa2; background: #f5eef8; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #7b1fa2; letter-spacing: .02em;">📝 Grounding SOP 변경 실습</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">6. 수정 SOP의 새 A2A 계획 확인</h2><p style="margin: 7px 0;">인덱싱이 끝난 뒤 이 셀만 실행하세요. 새 A2A 대화를 시작해 첫 <code>Orchestrator Decision</code>까지만 확인합니다. 앞선 셀의 실행 상태는 사용하지 않습니다.</p></div></div>


In [25]:
# A2A 실행은 위의 streaming handoff 예제를 사용합니다.
# print('문서 근거:', task_plan.get('grounding_basis', '없음'))
# print('계획:', task_plan.get('summary', '없음'))
# for number, task in enumerate(task_plan.get('planned_tasks', []), start=1):
#     print(f'{number}. {task}')

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #ef6c00; background: #fff3e0; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #ef6c00; letter-spacing: .02em;">↩️ 실습 종료 후 원복</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">수정 SOP로 UI 요청을 확인한 뒤 다음 셀을 실행하면, <code>data/context/collaboration_sop.md</code> 원본 파일을 Grounding 저장소에 다시 업로드합니다.</div></div>


In [26]:
restore_result = sop_grounding.restore_sop(sop_change)
print("원본 SOP 파일의 Grounding 인덱싱을 다시 요청했습니다:", restore_result)
print("인덱싱 반영 뒤에는 원래 SOP 기준으로 UI 요청을 보낼 수 있습니다.")

원본 SOP 파일의 Grounding 인덱싱을 다시 요청했습니다: {'documents': [{'metadata': [{'key': 'id', 'value': ['collaboration_sop.md'], 'matchMode': 'ANY'}], 'id': 'b5656983-0e19-42c8-9717-e520e5a424b1'}]}
인덱싱 반영 뒤에는 원래 SOP 기준으로 UI 요청을 보낼 수 있습니다.


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">🔎 코드 리뷰 · 공개 ReAct 실행 로그</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">7. Orchestration Agent의 첫 판단 흐름 확인</h2><p style="margin: 7px 0;">공개된 <b>Field handoff → Observation → (필요 시 Grounding Tool) → 첫 Decision</b>만 독립적으로 실행합니다. 첫 판단 뒤에는 멈추므로 Team 실행·후속 handoff를 기다리지 않습니다. 모델의 비공개 사고 과정은 출력하지 않습니다.</p><p style="margin: 7px 0 0; color:#4b5f78;">로컬 A2A 서버가 실행 중이어야 합니다.</p></div></div>


In [27]:
from src.agent_to_agent.catalog import DEFAULT_REQUEST
from src.agent_to_agent.orchestration import run_first_react_turn

react_trace_steps = run_first_react_turn(DEFAULT_REQUEST)
print('첫 ReAct 턴의 공개 이벤트를 수집했습니다. 다음 셀에서 확인하세요.')

첫 ReAct 턴의 공개 이벤트를 수집했습니다. 다음 셀에서 확인하세요.


In [28]:
print('=== Orchestration Agent · First ReAct Turn ===')
for event in react_trace_steps:
    step = event['step']
    if step == 'field_handoff':
        print('\n[Input] ', event['handoff'].get('request'))
    elif step == 'observation':
        observation = event['observation']
        print('\n[Observe]')
        print('  상태:', observation.get('chat_summary'))
        print('  미해결:', '; '.join(observation.get('unresolved_decisions', [])))
        print('  다음 판단:', '; '.join(observation.get('decision_focus', [])))
    elif step == 'grounding':
        audit = event['audit']
        print('\n[Act · Grounding] ', f"{audit.get('state')} · {audit.get('document_count', 0)} documents")
    elif step == 'decision':
        decision = event['decision']
        print('\n[Decide] ', f"{decision.get('action')} → {decision.get('target_agent_id')}")
        print('  목적:', decision.get('purpose', '-'))
        print('  근거:', decision.get('reason'))

=== Orchestration Agent · First ReAct Turn ===

[Input]  CIP 펌프 메커니컬 실 키트를 구매해 주세요.

[Observe]
  상태: CIP 펌프용 메커니컬 실 키트 구매 요청을 접수했으며, 호환품 확인을 위해 펌프 식별정보와 규격·수량·납기·예산 조건이 필요합니다.
  미해결: 구매 검토에 필요한 펌프 식별정보, 규격, 수량, 납기 및 예산·승인 조건이 확인되지 않았습니다.
  다음 판단: 정확한 호환품과 구매 가능 여부를 판단할 수 있는 설비·자재 정보 확보가 우선입니다.

[Act · Grounding]  no_result · 23 documents

[Decide]  delegate → fb-facilities-team
  목적: CIP 펌프 메커니컬 실 키트의 호환품 판단에 필요한 설비·기술 정보를 구조화해 주세요.
  근거: 현재 펌프 식별정보와 실 규격·수량이 없어 구매 대상의 정확한 호환성과 구매 가능 여부를 먼저 확인해야 합니다.
